# Folds Inside Crystals: a Right-Angle Prism

A mirror surface next to a tensor material is a fold inside the crystal:
the hypotenuse of a right-angle prism (total internal reflection), a Porro
face, or a metal on a crystal face. List it as any Optiland mirror, with
`material="mirror"` and `interaction_type="anisotropic"`. The crystal is then
on both sides of the surface, and the next surface sees it as its incident
medium. `far_material` of the model is the medium behind the face: None for
air (a bare face), a scalar or tensor material (a metal), or
`"perfect_conductor"`.

The branch key entry of a fold is `(label, "F", mode)`: a reflection that
keeps the listed order of the surfaces. Power that leaves through the fold
face is the ledger field `escaped`.

In [1]:
import numpy as np

from optiland.materials import IdealMaterial, UniaxialMaterial
from optiland.optic import Optic
from optiland.rays import PolarizationState
from optiland.raytrace.branches import BranchTracer

wavelength = 0.5893


def right_angle_prism(material, fold_angle_deg=45.0):
    """Entrance face at z = 0, the fold 5 mm behind it (tilted about x), the
    exit face 5 mm along the folded axis, and the image 5 mm behind that."""
    fold = np.radians(fold_angle_deg)
    turn = 2 * fold
    axis = np.array([0.0, np.sin(turn), -np.cos(turn)])  # the folded axis
    tilt = turn - np.pi  # the exit face is normal to the folded axis
    prism = Optic()
    prism.surfaces.add(index=0, radius=np.inf, thickness=np.inf)
    prism.surfaces.add(
        index=1, thickness=5.0, material=material, is_stop=True,
        interaction_type="anisotropic", comment="in",
    )  # fmt: skip
    prism.surfaces.add(
        index=2, x=0.0, y=0.0, z=5.0, rx=fold, material="mirror",
        interaction_type="anisotropic", comment="hyp",
    )  # fmt: skip
    p = np.array([0.0, 0.0, 5.0]) + 5.0 * axis
    prism.surfaces.add(
        index=3, x=0.0, y=float(p[1]), z=float(p[2]), rx=tilt,
        interaction_type="anisotropic", comment="out",
    )  # fmt: skip
    q = p + 5.0 * axis
    prism.surfaces.add(index=4, x=0.0, y=float(q[1]), z=float(q[2]), rx=tilt)
    prism.set_aperture(aperture_type="EPD", value=2.0)
    prism.fields.set_type(field_type="angle")
    prism.fields.add(y=0.0)
    prism.wavelengths.add(value=wavelength, is_primary=True)
    prism.surfaces[2].interaction_model.far_material = None  # air behind the face
    return prism


def polarized(prism, ex, ey):
    prism.updater.set_polarization(
        PolarizationState(is_polarized=True, Ex=ex, Ey=ey, phase_x=0.0, phase_y=0.0)
    )


def face(n):
    """The transmittance 4n / (1 + n)² of a face at normal incidence."""
    return 4 * n / (1 + n) ** 2

## Calcite: total internal reflection

The optic axis is along x, normal to the plane of incidence (y–z). An x
input (s at the fold) is the e wave and sees n_e; a y input is the o wave
and sees n_o. Both indices are above √2, so both waves reflect totally at
45°: each branch carries the transmittance of the two faces, and nothing
escapes.

In [2]:
n_o, n_e = 1.6583434042, 1.4861300612  # calcite at 589.3 nm (Ghosh 1999)
calcite = UniaxialMaterial(
    IdealMaterial(n_o), IdealMaterial(n_e), optic_axis=(1.0, 0.0, 0.0)
)
prism = right_angle_prism(calcite)
polarized(prism, np.sqrt(0.5), np.sqrt(0.5))
result = BranchTracer(prism).trace_all(num_rays=1)
for key, branch in result.branches.items():
    print(f"{key}: power {branch.power:.12f}")
print(f"closed form: e {0.5 * face(n_e) ** 2:.12f}, o {0.5 * face(n_o) ** 2:.12f}")
print(f"escaped: {result.ledger.escaped:.1e}, total: {result.ledger.total():.15f}")

(('in', 'T', 'o'), ('hyp', 'F', 'o'), ('out', 'T')): power 0.440549378970
(('in', 'T', 'e'), ('hyp', 'F', 'e'), ('out', 'T')): power 0.462496281816
closed form: e 0.462496281816, o 0.440549378970
escaped: 0.0e+00, total: 1.000000000000000


Both branches of the chief ray leave along the folded axis (+y) from the
same point: with the optic axis normal to the plane of incidence, neither
mode walks off in that plane.

In [3]:
for key, branch in result.branches.items():
    r = branch.rays
    print(
        f"{key[0][2]}: exit point ({float(r.x[0]):+.6f}, {float(r.y[0]):+.6f},"
        f" {float(r.z[0]):+.6f}) mm, direction ({float(r.L[0]):+.6f},"
        f" {float(r.M[0]):+.6f}, {float(r.N[0]):+.6f})"
    )

o: exit point (+0.000000, +10.000000, +5.000000) mm, direction (+0.000000, +1.000000, +0.000000)
e: exit point (+0.000000, +10.000000, +5.000000) mm, direction (+0.000000, +1.000000, +0.000000)


## Quartz below the critical angle

Quartz (n_o = 1.5442, n_e = 1.5534) has a critical angle of about 40° to
air, so a fold at 30° reflects only part of the power. The reflected power is
the internal Fresnel R of the mode (s for the e wave, p for the o wave); the
rest leaves through the fold face and is booked as `escaped`.

In [4]:
q_o, q_e = 1.5442464, 1.5533580  # quartz at 589.3 nm (Ghosh 1999)
quartz = UniaxialMaterial(
    IdealMaterial(q_o), IdealMaterial(q_e), optic_axis=(1.0, 0.0, 0.0)
)
theta = np.radians(30.0)


def internal_fresnel(n, theta):
    """R_s, R_p from index n into air at the angle theta (below critical)."""
    c = np.cos(theta)
    ct = np.sqrt(1 - (n * np.sin(theta)) ** 2)
    rs = (n * c - ct) / (n * c + ct)
    rp = (c - n * ct) / (c + n * ct)
    return rs**2, rp**2


for (ex, ey), mode, n, which in (((1.0, 0.0), "e", q_e, 0), ((0.0, 1.0), "o", q_o, 1)):
    prism = right_angle_prism(quartz, fold_angle_deg=30.0)
    polarized(prism, ex, ey)
    result = BranchTracer(prism).trace_all(num_rays=1)
    key = (("in", "T", mode), ("hyp", "F", mode), ("out", "T"))
    R = internal_fresnel(n, theta)[which]
    print(
        f"{mode}: reflected branch {result[key].power:.12f}"
        f" (closed form {face(n) ** 2 * R:.12f});"
        f" escaped {result.ledger.escaped:.12f}"
        f" (closed form {face(n) * (1 - R):.12f})"
    )

e: reflected branch 0.119139816808 (closed form 0.119139816808); escaped 0.828022252929 (closed form 0.828022252929)
o: reflected branch 0.003547197968 (closed form 0.003547197968); escaped 0.950524096984 (closed form 0.950524096984)
